# Stacking ensemble machine learning models to improve forecasting

In machine learning, stacking is an ensemble technique that combines multiple models to reduce their biases and improve predictive performance. Specifically, the predictions of each model (base models) are stacked and used as input to a final model (metamodel) to compute the prediction.

Stacking is effective because it leverages the strengths of different algorithms and attempts to mitigate their individual weaknesses. By combining multiple models, it can capture complex patterns in the data and improve prediction accuracy.

However, stacking can be computationally expensive and requires careful tuning to avoid overfitting. To this end, it is highly recommended to train the final estimator through cross-validation. In addition, obtaining diverse and well-performing base models is critical to the success of the stacking technique.

With scikit-learn, it is very easy to combine multiple estimators thanks to the [StackingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html#sklearn.ensemble.StackingRegressor) class. The `estimators` parameter corresponds to the list of the estimators (base learners) that will be stacked in parallel on the input data. The `final_estimator` (metamodel) will use the predictions of the estimators as input.

<div class="admonition note" name="html-admonition" style="background: rgba(0,184,212,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00b8d4; border-color: #00b8d4; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00b8d4;"></i>
    <b style="color: #00b8d4;">✏️ Note</b>
</p>

See <a href="https://cienciadedatos.net/documentos/py52-stacking-ensemble-models-forecasting.html" target="_blank">Stacking (ensemble) machine learning models to improve forecasting</a> for a more detailed example of stacking models.

</div>

## Libraries and Data

In [1]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd

# Modelling and Forecasting
# ==============================================================================
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import KFold

from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import grid_search_forecaster

# Warnings
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Data
# ==============================================================================
data = fetch_dataset(name = 'fuel_consumption')
data = data.loc[:"2019-01-01", ['Gasolinas']]
data = data.rename(columns = {'Gasolinas': 'consumption'})
data.index.name = 'date'
data['consumption'] = data['consumption'] / 100000
data.head(3)

╭──────────────────────────────── fuel_consumption ────────────────────────────────╮
│ Description:                                                                     │
│ Monthly fuel consumption in Spain from 1969-01-01 to 2022-08-01.                 │
│                                                                                  │
│ Source:                                                                          │
│ Obtained from Corporación de Reservas Estratégicas de Productos Petrolíferos and │
│ Corporación de Derecho Público tutelada por el Ministerio para la Transición     │
│ Ecológica y el Reto Demográfico. https://www.cores.es/es/estadisticas            │
│                                                                                  │
│ URL:                                                                             │
│ https://raw.githubusercontent.com/skforecast/skforecast-                         │
│ datasets/main/data/consumos-combustibles-mensual.csv                             │
│                                                                                  │
│ Shape: 644 rows x 5 columns                                                      │
╰──────────────────────────────────────────────────────────────────────────────────╯

,consumption
date,
1969-01-01,1.668752
1969-02-01,1.554668
1969-03-01,1.849837


In addition to the past values of the series (lags), an additional variable indicating the month of the year is added. This variable is included in the model to capture the seasonality of the series.

In [3]:
# Calendar features
# ==============================================================================
data['month_of_year'] = data.index.month
data.head(3)

,consumption,month_of_year
date,,
1969-01-01,1.668752,1
1969-02-01,1.554668,2
1969-03-01,1.849837,3


To facilitate the training of the models, the search for optimal hyperparameters, and the evaluation of their predictive accuracy, the data are divided into three separate sets: training, validation, and test.

In [4]:
# Split train-validation-test
# ==============================================================================
end_train = '2007-12-01 23:59:00'
end_validation = '2012-12-01 23:59:00'
data_train = data.loc[: end_train, :]
data_val   = data.loc[end_train:end_validation, :]
data_test  = data.loc[end_validation:, :]

print(f"Dates train      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Dates validation : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Dates test       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Dates train      : 1969-01-01 00:00:00 --- 2007-12-01 00:00:00  (n=468)
Dates validation : 2008-01-01 00:00:00 --- 2012-12-01 00:00:00  (n=60)
Dates test       : 2013-01-01 00:00:00 --- 2019-01-01 00:00:00  (n=73)


## StackingRegressor

With scikit-learn it is very easy to combine multiple estimators thanks to the [StackingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html#sklearn.ensemble.StackingRegressor) class.

The `estimators` parameter corresponds to the list of the estimators (base learners) that will be stacked in parallel on the input data. It should be specified as a list of names and estimators. The `final_estimator` (metamodel) will use the predictions of the estimators as input.

In [5]:
# Create stacking estimator
# ==============================================================================
params_ridge = {'alpha': 0.001}
params_lgbm = {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500, 'verbose': -1}

estimators = [
    ('ridge', Ridge(**params_ridge)),
    ('lgbm', LGBMRegressor(random_state=42, **params_lgbm)),
]

stacking_estimator = StackingRegressor(
                         estimators = estimators,
                         final_estimator = Ridge(),
                         cv = KFold(n_splits=5, shuffle=False)
                     )
stacking_estimator

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('ridge', ...), ('lgbm', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",KFold(n_split...shuffle=False)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no imp

In [6]:
# Create forecaster
# ==============================================================================
forecaster = ForecasterRecursive(
                 estimator = stacking_estimator,
                 lags      = 12  # Last 12 months used as predictors
             )

In [7]:
# Backtesting on test data
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 12,  # Forecast horizon
         initial_train_size = len(data.loc[:end_validation]),
         refit              = False, 
     )

metric, predictions = backtesting_forecaster(
                          forecaster = forecaster,
                          y          = data['consumption'],
                          exog       = data['month_of_year'],
                          cv         = cv,
                          metric     = 'mean_squared_error',
                          n_jobs     = 'auto',
                          verbose    = False
                      )        

metric

  0%|          | 0/7 [00:00<?, ?it/s]

,mean_squared_error
0,0.053776


## Hiperparameters search of StackingRegressor

When using `StackingRegressor`, the hyperparameters of each estimator must be preceded by the name of the estimator followed by two underscores. For example, the `alpha` hyperparameter of the ridge estimator must be specified as `ridge__alpha`. The hyperparameter of the final estimator must be specified with the prefix `final_estimator__`.

In [8]:
# Grid search of hyperparameters and lags
# ==============================================================================
param_grid = {
    'ridge__alpha': [0.1, 1, 10],
    'lgbm__n_estimators': [100, 500],
    'lgbm__max_depth': [3, 5, 10],
    'lgbm__learning_rate': [0.01, 0.1],
    'final_estimator__alpha': [0.1, 1]
}

# Lags used as predictors
lags_grid = [24]

cv = TimeSeriesFold(
         steps              = 12,
         initial_train_size = len(data.loc[:end_train]),
         refit              = False, 
     )

results_grid = grid_search_forecaster(
                   forecaster  = forecaster,
                   y           = data['consumption'],
                   exog        = data['month_of_year'],
                   lags_grid   = lags_grid,
                   param_grid  = param_grid,
                   cv          = cv,
                   metric      = 'mean_squared_error'
               )

results_grid.head()

lags grid:   0%|          | 0/1 [00:00<?, ?it/s]

params grid:   0%|          | 0/72 [00:00<?, ?it/s]

,lags,lags_label,params,mean_squared_error,final_estimator__alpha,lgbm__learning_rate,lgbm__max_depth,lgbm__n_estimators,ridge__alpha
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{'final_estimator__alpha': 1, 'lgbm__learning_...",0.066354,1.0,0.01,3.0,100.0,0.1
1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{'final_estimator__alpha': 1, 'lgbm__learning_...",0.068780,1.0,0.01,3.0,100.0,1.0
2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{'final_estimator__alpha': 0.1, 'lgbm__learnin...",0.069371,0.1,0.01,3.0,100.0,0.1
3,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{'final_estimator__alpha': 1, 'lgbm__learning_...",0.069450,1.0,0.10,10.0,500.0,0.1
4,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{'final_estimator__alpha': 1, 'lgbm__learning_...",0.069464,1.0,0.01,5.0,100.0,0.1


Once the best hyperparameters have been determined for each estimator in the ensemble, the test error is computed through backtesting.

In [9]:
# Backtesting on test data
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 12,
         initial_train_size = len(data.loc[:end_validation]),
         refit              = False, 
     )

metric, predictions = backtesting_forecaster(
                          forecaster = forecaster,
                          y          = data['consumption'],
                          exog       = data['month_of_year'],
                          cv         = cv,
                          metric     = 'mean_squared_error'
                      )        

metric

  0%|          | 0/7 [00:00<?, ?it/s]

,mean_squared_error
0,0.013506


## Feature importance in StackingRegressor

When a estimator of type `StackingRegressor` is used as a estimator in a predictor, its `get_feature_importances` method will not work. This is because objects of type `StackingRegressor` do not have either the `feature_importances` or `coef_` attribute. Instead, it is necessary to inspect each of the estimators that are part of the stacking.

In [10]:
# Feature importances for each estimator in the stacking
# ==============================================================================
if forecaster.estimator.__class__.__name__ == 'StackingRegressor':
    importance_pred = []
    for estimator in forecaster.estimator.estimators_:
        try:
            importance = pd.DataFrame(
                data = {
                    'feature': forecaster.estimator.feature_names_in_,
                    f'importance_{type(estimator).__name__}': estimator.coef_,
                    f'importance_abs_{type(estimator).__name__}': np.abs(estimator.coef_)
                }
            ).set_index('feature')
        except:
            importance = pd.DataFrame(
                data = {
                    'feature': forecaster.estimator.feature_names_in_,
                    f'importance_{type(estimator).__name__}': estimator.feature_importances_,
                    f'importance_abs_{type(estimator).__name__}': np.abs(estimator.feature_importances_)
                }
            ).set_index('feature')
        importance_pred.append(importance)
    
    importance_pred = pd.concat(importance_pred, axis=1)
    
else:
    importance_pred = forecaster.get_feature_importances()
    importance_pred['importance_abs'] = importance_pred['importance'].abs()
    importance_pred = importance_pred.sort_values(by='importance_abs', ascending=False)

importance_pred.head(5)

,importance_Ridge,importance_abs_Ridge,importance_LGBMRegressor,importance_abs_LGBMRegressor
feature,,,,
Column_0,0.020984,0.020984,59,59
Column_1,0.216998,0.216998,1,1
Column_2,0.188519,0.188519,0,0
Column_3,0.200916,0.200916,0,0
Column_4,0.106734,0.106734,0,0
